# 03 - LSTM residual corrector vs SARIMAX vs naive

Honest A/B/C comparison. Walk-forward, per-origin alignment.

- **A. Naive** (last-value baseline)
- **B. SARIMAX(2,1,2)** (current production)
- **C. SARIMAX + LSTM residual corrector** (proposed hybrid)

Decision criteria (printed at the end):
- Hybrid (C) MAE < SARIMAX (B) MAE by >=10%
- LSTM R^2 on residuals >= 0.10
- Both met -> DEPLOY hybrid; else stay on SARIMAX-only.

In [ ]:
%pip install -q -r ../requirements-notebooks.txt

In [ ]:
import json, sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from statsmodels.tsa.statespace.sarimax import SARIMAX

from lib.preprocess import build_wide, load_csv_dir
from lib.lstm import HORIZONS, LOOKBACK, ResidualLSTM, train_residual_lstm

DATASET_DIR = Path.cwd().parent.parent / 'dataset'
wide = build_wide(load_csv_dir(DATASET_DIR))
ORDER = tuple(json.load((Path.cwd().parent / 'sarimax_config.json').open())['order'])
lags = json.load((Path.cwd().parent / 'lags.json').open()) if (Path.cwd().parent / 'lags.json').exists() else {}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'wide={wide.shape}  order={ORDER}  horizons={HORIZONS}  device={DEVICE}')

## Build aligned y, exog (for SARIMAX), and 6-feature matrix (for LSTM)

In [ ]:
def build_features(wide, lags):
    feats = pd.DataFrame(index=wide.index)
    feats['centro_flow'] = wide['station-01_flow_rate']
    for col, default_lag in [('station-02_water_level', 5), ('station-02_flow_rate', 5), ('station-03_water_level', 8), ('station-03_flow_rate', 8)]:
        L = max(int(lags.get(col, {}).get('lag_min', default_lag)), 0)
        feats[f'{col}_lag{L}'] = wide[col].shift(L)
    return feats.dropna()

y = wide['station-01_water_level'].copy()
X_exog = build_features(wide, lags)
common = y.index.intersection(X_exog.index)
y = y.loc[common].dropna(); X_exog = X_exog.loc[y.index]
X_exog = X_exog[X_exog.columns[X_exog.std() > 1e-9]]

lstm_features = pd.DataFrame({
    'centro_level':  wide['station-01_water_level'],
    'centro_flow':   wide['station-01_flow_rate'],
    'napo_level':    wide['station-02_water_level'],
    'napo_flow':     wide['station-02_flow_rate'],
    'balinad_level': wide['station-03_water_level'],
    'balinad_flow':  wide['station-03_flow_rate'],
}).reindex(y.index).ffill().dropna()

common = y.index.intersection(lstm_features.index)
y = y.loc[common]; X_exog = X_exog.loc[common]; lstm_features = lstm_features.loc[common]
y_arr = y.values
feat_arr = lstm_features.values.astype(np.float32)
N = len(y)
print(f'aligned: y={y.shape}  X_exog={X_exog.shape}  feat={feat_arr.shape}')

## Walk-forward: per-origin records

For each origin t step ORIGIN_STEP: refit SARIMAX on y[:t], forecast horizons {10,30,60} -> sarimax[h]; record actual y[t+h-1], residual = actual - sarimax, LSTM input feat[t-LOOKBACK:t].

Causal alignment: LSTM input never peeks at the future.

In [ ]:
MAX_H = max(HORIZONS)
ORIGIN_STEP = 5
MIN_ORIGIN = max(LOOKBACK + 100, 200)
MAX_ORIGIN = N - MAX_H - 1
origins = list(range(MIN_ORIGIN, MAX_ORIGIN, ORIGIN_STEP))
print(f'origins to evaluate: {len(origins)}')

records = []
for t in origins:
    try:
        m = SARIMAX(y.iloc[:t], exog=X_exog.iloc[:t], order=ORDER,
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=30)
        future_X = X_exog.iloc[t : t + MAX_H]
        if len(future_X) < MAX_H: continue
        fc = np.asarray(m.get_forecast(steps=MAX_H, exog=future_X).predicted_mean)
        sarimax = np.array([fc[h - 1] for h in HORIZONS])
        actual  = np.array([y_arr[t + h - 1] for h in HORIZONS])
        records.append({
            't': t,
            'seq': feat_arr[t - LOOKBACK : t],
            'sarimax': sarimax,
            'actual': actual,
            'residual': actual - sarimax,
            'naive': np.full(len(HORIZONS), y_arr[t - 1]),
        })
    except Exception:
        continue
print(f'valid records: {len(records)}')

## Chronological split (60/20/20) and standardize features on train stats only

In [ ]:
n = len(records); n_train = int(n * 0.6); n_val = int(n * 0.2)
train, val, test = records[:n_train], records[n_train:n_train + n_val], records[n_train + n_val:]
print(f'split: train={len(train)}  val={len(val)}  test={len(test)}')

stack = lambda recs, key: np.stack([r[key] for r in recs])
X_tr = stack(train, 'seq'); Y_tr = stack(train, 'residual').astype(np.float32)
X_va = stack(val,   'seq'); Y_va = stack(val,   'residual').astype(np.float32)
X_te = stack(test,  'seq'); Y_te = stack(test,  'residual').astype(np.float32)

mu = X_tr.reshape(-1, X_tr.shape[-1]).mean(axis=0)
sd = X_tr.reshape(-1, X_tr.shape[-1]).std(axis=0); sd[sd < 1e-6] = 1.0
X_tr_z = ((X_tr - mu) / sd).astype(np.float32)
X_va_z = ((X_va - mu) / sd).astype(np.float32)
X_te_z = ((X_te - mu) / sd).astype(np.float32)

print('train residual std/horizon:', [f'{Y_tr[:, j].std():.3f}' for j in range(len(HORIZONS))])

## Train LSTM residual corrector

In [ ]:
torch.manual_seed(42); np.random.seed(42)
model, hist = train_residual_lstm(X_tr_z, Y_tr, X_va_z, Y_va, device=DEVICE, epochs=300, patience=30)
with torch.no_grad():
    pred_res = model(torch.tensor(X_te_z, device=DEVICE)).cpu().numpy()
print(f'final val_loss = {hist["val"][-1]:.4f}  (best train_loss = {min(hist["train"]):.4f})')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(hist['train'], label='train', color='#4fc3f7')
ax.plot(hist['val'],   label='val',   color='#ff7043')
ax.set_xlabel('epoch'); ax.set_ylabel('MSE loss'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('LSTM training history')
plt.tight_layout()

## Compute MAE for all 3 models

In [ ]:
actual_te  = stack(test, 'actual')
sarimax_te = stack(test, 'sarimax')
naive_te   = stack(test, 'naive')
hybrid_te  = sarimax_te + pred_res

mae_per_h = lambda a, b: [float(np.abs(a[:, j] - b[:, j]).mean()) for j in range(a.shape[1])]
naive_mae   = mae_per_h(actual_te, naive_te)
sarimax_mae = mae_per_h(actual_te, sarimax_te)
hybrid_mae  = mae_per_h(actual_te, hybrid_te)

results = pd.DataFrame({
    'model':   ['A. Naive', 'B. SARIMAX', 'C. SARIMAX + LSTM (hybrid)'],
    f'MAE@{HORIZONS[0]}': [naive_mae[0], sarimax_mae[0], hybrid_mae[0]],
    f'MAE@{HORIZONS[1]}': [naive_mae[1], sarimax_mae[1], hybrid_mae[1]],
    f'MAE@{HORIZONS[2]}': [naive_mae[2], sarimax_mae[2], hybrid_mae[2]],
})
results['MAE_avg'] = results[[c for c in results.columns if 'MAE@' in c]].mean(axis=1)
results.round(3)

## Did the LSTM learn anything? (R^2 on residuals - the honesty check)

In [ ]:
ss_res = ((Y_te - pred_res) ** 2).sum(axis=0)
ss_tot = ((Y_te - Y_te.mean(axis=0)) ** 2).sum(axis=0)
r2 = 1 - ss_res / np.maximum(ss_tot, 1e-9)
for j, h in enumerate(HORIZONS):
    verdict = 'learned signal' if r2[j] >= 0.1 else 'no real learning' if r2[j] >= -0.1 else 'WORSE than mean residual'
    print(f'R^2 @ {h:>3} min: {r2[j]:+.3f}  ({verdict})')
r2_avg = float(r2.mean())
print(f'\naverage R^2 = {r2_avg:+.3f}')

## Decision

In [ ]:
sarimax_avg = float(np.mean(sarimax_mae))
hybrid_avg  = float(np.mean(hybrid_mae))
improvement = (sarimax_avg - hybrid_avg) / sarimax_avg * 100

print(f'SARIMAX MAE_avg : {sarimax_avg:.3f} cm')
print(f'Hybrid  MAE_avg : {hybrid_avg:.3f} cm   (delta = {improvement:+.1f}%)')
print(f'LSTM R^2_avg    : {r2_avg:+.3f}')

deploy = (improvement >= 10.0) and (r2_avg >= 0.1)
print()
print('=' * 50)
if deploy:
    print('DECISION: DEPLOY hybrid')
else:
    print('DECISION: STAY ON SARIMAX-only')
    print(f'  reason: improvement={improvement:+.1f}% (need >=10%), R^2={r2_avg:+.2f} (need >=0.10)')
print('=' * 50)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ts = [r['t'] for r in test]
h_pick = 1   # 30-min horizon
ax.plot(ts, actual_te[:, h_pick],  color='#4fc3f7', lw=1.5, label='actual')
ax.plot(ts, sarimax_te[:, h_pick], color='#ff7043', lw=1.0, ls='--', label='SARIMAX')
ax.plot(ts, hybrid_te[:, h_pick],  color='#ab47bc', lw=1.0, ls='--', label='SARIMAX+LSTM hybrid')
ax.plot(ts, naive_te[:, h_pick],   color='#888',    lw=0.7,             label='naive')
ax.set_title(f'30-min ahead Centro forecast (test set, n={len(test)})')
ax.set_xlabel('time index (origin)'); ax.set_ylabel('water level (cm)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## Save artifacts (decision file always; weights only if DEPLOY)

In [ ]:
out_path = Path.cwd().parent
decision = {
    'deploy': bool(deploy),
    'sarimax_mae_avg_cm': sarimax_avg,
    'hybrid_mae_avg_cm':  hybrid_avg,
    'naive_mae_avg_cm':   float(np.mean(naive_mae)),
    'improvement_pct':    improvement,
    'r2_avg':             r2_avg,
    'r2_per_horizon':     [float(v) for v in r2],
    'sarimax_mae_per_horizon': [float(v) for v in sarimax_mae],
    'hybrid_mae_per_horizon':  [float(v) for v in hybrid_mae],
    'naive_mae_per_horizon':   [float(v) for v in naive_mae],
    'horizons_min':       list(HORIZONS),
    'feature_means':      mu.tolist(),
    'feature_stds':       sd.tolist(),
    'sarimax_order':      list(ORDER),
    'lookback':           LOOKBACK,
    'n_train':            len(train),
    'n_test':             len(test),
}
(out_path / 'lstm_decision.json').write_text(json.dumps(decision, indent=2))
print(f'wrote {out_path / "lstm_decision.json"}')

if deploy:
    torch.save(model.state_dict(), out_path / 'lstm_residual.pt')
    print(f'wrote {out_path / "lstm_residual.pt"}')
else:
    print('no .pt saved - predict.py continues using SARIMAX-only')